# Pipeline ETL Immobilier — Extraction, Transformation, KPI & Load

Ce notebook répond à l'exigence du cahier des charges imposant une présentation sous
forme de notebook Jupyter. Il reprend fidèlement le pipeline ETL réel de
`app/pages/2_ETL_demo.py` (extraction des 8 sources brutes, transformation, calcul des
KPI, chargement en base PostgreSQL).

Toute la logique métier vit dans `src/` (`extract.py`, `transform.py`, `API.py`,
`load.py`) : ce notebook se contente d'orchestrer ces modules et d'afficher leurs
résultats, exactement comme le fait l'application Streamlit — aucune règle de gestion
n'est dupliquée ici.

Ce notebook écrit ses résultats dans `data/final/*.csv`. La partie **présentation** du
dashboard (KPIs, Top 10, cartes) est dans le notebook séparé
[`dashboard.ipynb`](dashboard.ipynb), qui relit ces mêmes CSV — lance donc celui-ci en
premier (ou au moins une fois) avant d'ouvrir `dashboard.ipynb`.

**Question métier du projet** : dans quelle ville française acheter pour investir ?

`score_attractivite` = 35% rendement brut + 25% ratio d'effort fiscal (inversé)
+ 20% revenu fiscal moyen + 20% taux de vacance (inversé), chaque composante
normalisée de 0 à 100 au niveau national.

## 0. Configuration

Si le kernel `immobilier` n'apparaît pas dans Jupyter, enregistre-le une fois
(environnement conda actif) :

```bash
python -m ipykernel install --user --name immobilier --display-name "Python (immobilier)"
```

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Localise la racine du projet (ce notebook peut être lancé depuis notebooks/ ou
# depuis la racine selon la façon dont Jupyter est démarré).
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import API
import extract
import load as load_module
import transform

pd.set_option("display.max_columns", 30)

# True  : exécute le vrai pipeline (extraction réseau des 8 sources + transformation
#         + KPI), comme le bouton "Lancer le pipeline complet" de 2_ETL_demo.py.
# False : ignore l'extraction réseau et relit directement data/final/*.csv, déjà
#         calculés par une exécution précédente (utile hors-ligne).
RUN_LIVE_PIPELINE = True

## 1. EXTRACT — lecture des 8 sources brutes

`transactions.npz` (historique local), DVF 2025 (data.gouv.fr), Carte des loyers
2024/2025, IRCOM (revenus des ménages, DGFiP), LOVAC (logements vacants, Cerema),
référentiel des communes (API geo.api.gouv.fr), IRL (API SDMX INSEE) et 3 séries
Banque de France (taux d'intérêt, flux d'emprunts, taux d'endettement). Mêmes appels
que `app/pages/2_ETL_demo.py`.

In [ ]:
if RUN_LIVE_PIPELINE:
    try:
        df_transactions_raw = extract.extract_transactions_npz()
        print(f"transactions.npz           : {len(df_transactions_raw):,} lignes brutes")

        df_dvf_2025 = extract.extract_dvf(2025)
        print(f"DVF 2025 (data.gouv.fr)    : {len(df_dvf_2025):,} lignes")

        loyers_2024 = extract.extract_loyers_complement(2024)
        loyers_2025 = extract.extract_loyers_complement(2025)
        print(f"Carte des loyers 2024+2025 : {len(loyers_2024) + len(loyers_2025):,} lignes")

        ircom_df = extract.extract_ircom()
        print(f"IRCOM (revenus, DGFiP)     : {len(ircom_df):,} lignes")

        lovac_df = extract.extract_lovac()
        print(f"LOVAC (logements vacants)  : {len(lovac_df):,} communes")

        communes_api = API.recuperer_toutes_communes()
        print(f"Référentiel communes (API) : {len(communes_api):,} communes")

        irl_df = API.recuperer_irl()
        ti_df = extract.extract_webstat_series(PROJECT_ROOT / "data/raw/additional_data/new_housing_loans_interest_rate.csv")
        flux_df = extract.extract_webstat_series(PROJECT_ROOT / "data/raw/additional_data/new_housing_loans_flow.csv")
        debt_df = extract.extract_webstat_series(PROJECT_ROOT / "data/raw/additional_data/household_debt_ratio.csv")
        print(f"IRL + 3 séries Banque de France : {len(irl_df)} trimestres, "
              f"{len(ti_df) + len(flux_df) + len(debt_df)} observations")

        print("\n\u2713 Extraction terminée.")
    except Exception as e:
        print(f"\u26a0 Échec de l'extraction en direct ({e!r}).")
        print("  Bascule sur les CSV déjà calculés dans data/final/.")
        RUN_LIVE_PIPELINE = False
else:
    print("RUN_LIVE_PIPELINE = False : extraction réseau ignorée, utilisation de data/final/*.csv.")

## 2. TRANSFORM — nettoyage & harmonisation

Fusion DVF 2025 + filtre qualité des transactions, harmonisation loyers/foyers
fiscaux/parc immobilier avec leurs compléments (Carte des loyers, IRCOM, LOVAC),
consolidation des indicateurs macro et construction du référentiel communes/démographie.

In [ ]:
if RUN_LIVE_PIPELINE:
    dvf_supplement = transform.transform_dvf_supplement(
        df_dvf_2025, id_transaction_offset=int(df_transactions_raw["id_transaction"].max()) + 1
    )
    transactions = transform.transform_transactions(
        pd.concat([df_transactions_raw, dvf_supplement], ignore_index=True)
    )
    print(f"transactions      : {len(transactions):,} lignes retenues (dont 2025)")

    loyers = transform.transform_loyers(pd.read_csv(PROJECT_ROOT / "data/raw/loyers.csv"), [loyers_2024, loyers_2025])
    foyers = transform.transform_foyers_fiscaux(pd.read_csv(PROJECT_ROOT / "data/raw/foyers_fiscaux.csv"), ircom_df)
    parc = transform.transform_parc_immobilier(pd.read_csv(PROJECT_ROOT / "data/raw/parc_immobilier.csv"), lovac_df)
    print(f"loyers            : {len(loyers):,} | foyers_fiscaux : {len(foyers):,} | parc_immobilier : {len(parc):,}")

    macro = transform.transform_indicateurs_macro(ti_df, flux_df, debt_df, irl_df)
    communes = transform.build_communes(communes_api)
    demographics = transform.build_demographics(communes_api)
    print(f"indicateurs_macro : {len(macro):,} mois | communes : {len(communes):,} | demographics : {len(demographics):,}")

    print("\n\u2713 Transformation terminée.")
else:
    final_dir = PROJECT_ROOT / "data" / "final"
    transactions = pd.read_csv(final_dir / "transactions.csv", parse_dates=["date_transaction"])
    loyers = pd.read_csv(final_dir / "loyers.csv")
    foyers = pd.read_csv(final_dir / "foyers_fiscaux.csv")
    parc = pd.read_csv(final_dir / "parc_immobilier.csv")
    macro = pd.read_csv(final_dir / "indicateurs_macro.csv")
    communes = pd.read_csv(final_dir / "communes.csv")
    demographics = pd.read_csv(final_dir / "demographics.csv")
    print(f"Rechargé depuis {final_dir} : transactions={len(transactions):,}, communes={len(communes):,}")

Aperçu des tables transformées :

In [ ]:
display(transactions.head(3))
display(communes.head(3))
display(loyers.head(3))

## 3. CALCUL DES KPI & SCORE D'ATTRACTIVITÉ

Rendement brut, taux de vacance, ratio d'effort fiscal, prix m² médian, puis
normalisation 0-100 et pondération pour obtenir `score_attractivite`
(cf. `transform.compute_kpi`).

In [ ]:
score = transform.compute_kpi(transactions, loyers, foyers, parc)
top = score.dropna(subset=["score_attractivite"]).sort_values("score_attractivite", ascending=False).iloc[0]
print(f"score_attractivite : {len(score):,} communes évaluées")
print(f"Année de référence  : {score['annee_ref'].iloc[0]}")
print(f"Meilleur score      : id_ville {int(top['id_ville'])} ({top['score_attractivite']:.1f}/100)")
score.sort_values("score_attractivite", ascending=False).head(10)

## 4. LOAD — écriture CSV + chargement PostgreSQL (optionnel)

Écrit un CSV par table dans `data/final/` (toujours — c'est ce que relit
`dashboard.ipynb`), puis tente le chargement dans PostgreSQL via `src/load.py` (même
verrou `pg_advisory_lock` que l'app Streamlit, pour éviter un chargement en double si
l'app tourne en parallèle). Si Docker/PostgreSQL n'est pas démarré, le chargement en
base est simplement ignoré.

In [ ]:
valid_ids = set(communes["id_ville"])
outputs = {
    "communes": communes,
    "demographics": demographics,
    "transactions": transactions[transactions["id_ville"].isin(valid_ids)],
    "loyers": loyers[loyers["id_ville"].isin(valid_ids)],
    "foyers_fiscaux": foyers[foyers["id_ville"].isin(valid_ids)],
    "parc_immobilier": parc[parc["id_ville"].isin(valid_ids)],
    "indicateurs_macro": macro,
    "score_attractivite": score[score["id_ville"].isin(valid_ids)],
}

load_module.DATA_DIR.mkdir(parents=True, exist_ok=True)
for table_name, table_df in outputs.items():
    table_df.to_csv(load_module.DATA_DIR / f"{table_name}.csv", index=False)
print(f"CSV écrits dans {load_module.DATA_DIR}")

PIPELINE_LOCK_KEY = 918273  # même clé que app/pages/2_ETL_demo.py et src/load.py

try:
    lock_conn = load_module.engine.connect()
    acquired = lock_conn.execute(
        load_module.text("SELECT pg_try_advisory_lock(:key)"), {"key": PIPELINE_LOCK_KEY}
    ).scalar()

    if not acquired:
        print("\u26a0 Un chargement est déjà en cours (verrou actif) — chargement PostgreSQL ignoré.")
        lock_conn.close()
    else:
        try:
            with load_module.engine.begin() as conn:
                for table_name in load_module.TABLES:
                    conn.execute(load_module.text(f"TRUNCATE operationnel.{table_name} CASCADE"))
            for table_name in load_module.TABLES:
                load_module.load_table(table_name)
            print("\n\u2713 Base PostgreSQL synchronisée.")
        finally:
            lock_conn.execute(load_module.text("SELECT pg_advisory_unlock(:key)"), {"key": PIPELINE_LOCK_KEY})
            lock_conn.close()
except Exception as e:
    print(f"\u26a0 Chargement PostgreSQL ignoré (base non accessible) : {e!r}")

---
*Pipeline terminé — `data/final/*.csv` est à jour. Ouvre
[`dashboard.ipynb`](dashboard.ipynb) pour la présentation (KPIs, Top 10, cartes).*